# Sensor Modalities: Camera, LiDAR and Radar

**Module 01 · Perception Systems** · Notebook 2 of 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/01_Perception/02_sensor_modalities.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Beginner | ~60 min | [01 SAE levels and ODD](01_sae_levels_and_odd.ipynb) | ISO 21448:2022 (context) |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Describe the operating principles of camera, LiDAR and radar (incl. 4D imaging radar and FMCW LiDAR)
2. Compare sensor capabilities and limitations across weather and lighting conditions
3. Generate and visualise a ray-cast LiDAR scan, its bird's-eye view (BEV) and a matching camera view
4. Compute the radar Doppler shift and explain why radar measures velocity directly
5. Quantify redundancy with and without common-cause failures
6. Explain why sensor diversity supports, but does not by itself establish, a safety argument

## Setup

Only NumPy, pandas and Matplotlib are required; Plotly is used for an optional interactive 3D view if available.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

rng = np.random.default_rng(42)
OKABE_ITO = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#000000']
SENSOR_COLOURS = {'Camera': OKABE_ITO[0], 'LiDAR': OKABE_ITO[2], 'Radar': OKABE_ITO[4], 'Fusion': OKABE_ITO[6]}
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
print(f"NumPy {np.__version__}")

---
## Part 1: Sensor comparison

Order-of-magnitude characteristics of automotive sensors (exact figures depend on the product, target reflectivity and conditions):

In [ ]:
sensor_table = pd.DataFrame({
    'Sensor': ['Camera', 'LiDAR', 'Radar (conventional)', '4D imaging radar'],
    'Measures': ['Colour/intensity per pixel (2D)', 'Range + angle per point (3D)',
                 'Range, radial velocity, azimuth', 'Range, radial velocity, azimuth, elevation'],
    'Typical range': ['Optics-dependent (~100 m+)', '~150-250 m at 10 % reflectivity (long-range units)',
                      '~200-300 m (long-range)', '~200-300 m (long-range)'],
    'Angular resolution': ['High (megapixels)', 'High (~0.1-0.2 deg)', 'Low (several deg)', 'Medium (~1 deg)'],
    'Weather robustness': ['Poor in glare, fog, heavy rain, darkness', 'Degrades in fog, heavy rain, snow, spray',
                           'Good', 'Good'],
    'Main use': ['Classification, signs, lanes', '3D detection, localisation, free space',
                 'Velocity, all-weather detection', 'All-weather detection with height'],
})
display(sensor_table)
print("--- Sensor comparison (text summary) ---")
for _, r in sensor_table.iterrows():
    print(f"  {r['Sensor']}: measures {r['Measures']}; range {r['Typical range']}; "
          f"resolution {r['Angular resolution']}; weather: {r['Weather robustness']}; use: {r['Main use']}")

### Camera
**Advantages:** high resolution (2–8 MP and more), rich semantics (colour, texture, text on signs), low cost, excellent for classification.
**Disadvantages:** no direct depth (monocular), sensitive to illumination (night, glare, tunnels) and to fog, rain and dirt on the lens.

### LiDAR (Light Detection and Ranging)
**Advantages:** accurate 3D measurement (cm-level), independent of ambient light, high angular resolution.
**Disadvantages:**
- Cost: from about US\$200 for 2025 automotive units (e.g. Hesai ATX, in mass production since 2025, [Reuters via LEDinside, Nov 2024](https://www.ledinside.com/node/34820)) up to tens of thousands of dollars for high-end or legacy spinning units (as of 2026).
- Degrades in fog, heavy rain, snow and spray (backscatter and attenuation).
- No colour; sparse at long range (point density falls roughly with 1/d², see [03 LiDAR and point clouds](03_lidar_point_clouds.ipynb)).
- Many designs have moving parts (spinning heads, polygon or MEMS mirrors); flash and optical-phased-array designs are fully solid-state.
- Range: ~150–250 m at 10 % reflectivity for long-range automotive units — always check the reflectivity at which a range is quoted.

Most LiDARs measure range by pulsed **time of flight**; **FMCW LiDAR** additionally measures radial velocity per point via the Doppler effect.

### Radar (Radio Detection and Ranging)
**Advantages:** robust in rain, fog and darkness; measures radial velocity **directly** (Doppler); long range; mature and low-cost.
**Disadvantages:**
- Low angular resolution; ghost targets from multipath reflections (e.g. guard rails, tunnels).
- Conventional automotive radar gives little or no elevation information (a bridge and a stopped truck can look similar); **4D imaging radar** adds elevation.
- Limited classification from sparse returns (micro-Doppler signatures and 4D radar improve this).

---
## Part 2: A ray-cast LiDAR scan

A spinning LiDAR fires a vertical fan of beams while rotating. Each beam returns the first surface it hits. We simulate this with simple geometry (axis-aligned boxes on a flat ground plane). Coordinates follow ISO 8855: **x forward, y left, z up**, origin at the sensor, mounted 1.8 m above the road.

> **Illustrative data:** the scene and sensor below are synthetic, chosen for teaching; they are not measurements of a real sensor.

In [ ]:
SENSOR_H = 1.8      # sensor height above ground [m]
G = -SENSOR_H       # ground plane z in the sensor frame

# Scene: axis-aligned boxes (xmin, xmax, ymin, ymax, zmin, zmax) in metres, sensor frame
SCENE = {
    'car_ahead':     ((10.0, 14.5, -1.0,  1.0), (G, G + 1.5)),
    'car_left':      ((25.0, 29.5,  2.5,  4.3), (G, G + 1.5)),
    'truck':         ((40.0, 48.0, -4.5, -2.0), (G, G + 3.5)),
    'pedestrian':    ((18.0, 18.5, -3.5, -3.0), (G, G + 1.75)),
    'building_left': ((5.0, 60.0, 10.0, 25.0), (G, G + 9.0)),
    'building_right':((5.0, 60.0, -25.0, -10.0), (G, G + 9.0)),
}
CLASS_OF = {'car_ahead': 'car', 'car_left': 'car', 'truck': 'truck', 'pedestrian': 'pedestrian',
            'building_left': 'building', 'building_right': 'building'}

def ray_boxes(origin, dirs, scene=SCENE, ground_z=G, max_range=80.0):
    """First-hit distance of each ray against all boxes and the ground plane.
    Returns (t, object_index) with t = inf for no hit; index -1 = ground, -2 = none."""
    n = len(dirs)
    t_best = np.full(n, np.inf); idx = np.full(n, -2)
    with np.errstate(divide='ignore', invalid='ignore'):
        inv = 1.0 / dirs
        for k, ((x0, x1, y0, y1), (z0, z1)) in enumerate(scene.values()):
            bmin = np.array([x0, y0, z0]); bmax = np.array([x1, y1, z1])
            t1 = (bmin - origin) * inv; t2 = (bmax - origin) * inv
            tmin = np.nanmax(np.minimum(t1, t2), axis=1); tmax = np.nanmin(np.maximum(t1, t2), axis=1)
            hit = (tmax >= np.maximum(tmin, 0)) & (tmin > 0) & (tmin < t_best)
            t_best[hit] = tmin[hit]; idx[hit] = k
        tg = (ground_z - origin[2]) / dirs[:, 2]
        hit = (dirs[:, 2] < 0) & (tg < t_best)
        t_best[hit] = tg[hit]; idx[hit] = -1
    out = t_best > max_range
    t_best[out] = np.inf; idx[out] = -2
    return t_best, idx

def simulate_spinning_lidar(n_beams=32, h_res_deg=0.2, v_fov=(-25.0, 10.0), range_noise=0.02, max_range=80.0):
    az = np.deg2rad(np.arange(0, 360, h_res_deg))
    el = np.deg2rad(np.linspace(v_fov[0], v_fov[1], n_beams))
    A, E = np.meshgrid(az, el)
    dirs = np.stack([np.cos(E) * np.cos(A), np.cos(E) * np.sin(A), np.sin(E)], -1).reshape(-1, 3)
    t, idx = ray_boxes(np.zeros(3), dirs, max_range=max_range)
    ok = np.isfinite(t)
    t_noisy = t[ok] + rng.normal(0, range_noise, ok.sum())
    pts = dirs[ok] * t_noisy[:, None]
    ring = np.repeat(np.arange(n_beams), len(az))[ok]
    return pts, idx[ok], ring, len(dirs)

points, labels, ring, n_rays = simulate_spinning_lidar()
names = list(SCENE.keys())
print(f"Rays fired: {n_rays:,}; returns: {len(points):,} ({100*len(points)/n_rays:.1f} %); "
      "the rest point at the sky or beyond max range")
print(f"X range [{points[:,0].min():.1f}, {points[:,0].max():.1f}] m, "
      f"Y range [{points[:,1].min():.1f}, {points[:,1].max():.1f}] m, "
      f"Z range [{points[:,2].min():.2f}, {points[:,2].max():.2f}] m")
print("Returns per object:")
print(f"  ground: {(labels == -1).sum():,}")
for k, nme in enumerate(names):
    print(f"  {nme}: {(labels == k).sum():,}")
print("Note: real LiDAR also reports an intensity/reflectivity value per point (not simulated here).")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sc = axes[0].scatter(points[:, 0], points[:, 1], c=points[:, 2], cmap='viridis', s=0.4)
axes[0].set_xlim(-60, 60); axes[0].set_ylim(-60, 60); axes[0].set_aspect('equal')
axes[0].set_xlabel('x, forward (m)'); axes[0].set_ylabel('y, left (m)')
axes[0].set_title("Top view (bird's eye)")
fig.colorbar(sc, ax=axes[0], label='Height z (m)')

front = (points[:, 0] > 0) & (np.abs(points[:, 1]) < 8)
axes[1].scatter(points[front, 0], points[front, 2], c=points[front, 1], cmap='cividis', s=0.4)
axes[1].axhline(G, color=OKABE_ITO[5], ls='--', label=f'Ground (z = {G:.1f} m)')
axes[1].set_xlabel('x, forward (m)'); axes[1].set_ylabel('z, up (m)'); axes[1].set_title('Side view (|y| < 8 m)')
axes[1].legend(loc='upper right')

axes[2].scatter(points[front, 1], points[front, 2], c=points[front, 0], cmap='cividis', s=0.4)
axes[2].axhline(G, color=OKABE_ITO[5], ls='--', label='Ground')
axes[2].invert_xaxis()
axes[2].set_xlabel('y, left (m)'); axes[2].set_ylabel('z, up (m)'); axes[2].set_title('Front view (x > 0, |y| < 8 m)')
axes[2].legend(loc='upper right')
fig.suptitle('Simulated 32-beam spinning LiDAR scan (synthetic scene)')
plt.tight_layout(); plt.show()

r = np.linalg.norm(points[:, :2], axis=1)
ground = labels == -1
n_near, n_far = (ground & (r >= 5) & (r < 10)).sum(), (ground & (r >= 40) & (r < 60)).sum()
a_near, a_far = np.pi * (10**2 - 5**2), np.pi * (60**2 - 40**2)
d_near, d_far = n_near / a_near, n_far / a_far
print("--- LiDAR projections (text summary) ---")
print("  Top view: concentric rings on the ground (one per beam), denser close to the sensor; empty 'shadows' behind objects.")
print(f"  Ground returns per m^2: {d_near:.1f} at 5-10 m vs {d_far:.3f} at 40-60 m "
      f"-> {d_near/max(d_far,1e-9):.0f}x denser close to the sensor.")
print(f"  Ground plane at z = {G:.1f} m (sensor frame).")

### Interactive 3D view

If Plotly is installed (it is on Colab), the next cell shows a rotatable 3D view; otherwise it falls back to a static Matplotlib view.

In [ ]:
sub = points[::4]
try:
    import plotly.graph_objects as go
    fig = go.Figure(go.Scatter3d(x=sub[:, 0], y=sub[:, 1], z=sub[:, 2], mode='markers',
                                 marker=dict(size=1.5, color=sub[:, 2], colorscale='Viridis',
                                             colorbar=dict(title='z (m)'))))
    fig.update_layout(title='Simulated LiDAR scan (drag to rotate)', height=550,
                      scene=dict(xaxis_title='x fwd (m)', yaxis_title='y left (m)', zaxis_title='z up (m)',
                                 aspectmode='data'))
    fig.show()
except ImportError:
    fig = plt.figure(figsize=(10, 7)); ax = fig.add_subplot(111, projection='3d')
    ax.scatter(sub[:, 0], sub[:, 1], sub[:, 2], c=sub[:, 2], cmap='viridis', s=0.5)
    ax.set_xlabel('x fwd (m)'); ax.set_ylabel('y left (m)'); ax.set_zlabel('z up (m)')
    ax.set_title('Simulated LiDAR scan (static view)'); plt.show()
print(f"3D view shows {len(sub):,} of {len(points):,} points (every 4th point).")

---
## Part 3: Bird's-eye view (BEV)

Many detectors and planners work on a **BEV grid**: the point cloud is discretised into cells on the ground plane, and each cell stores features (here: point count and maximum height). The axes are in metres.

In [ ]:
def birds_eye_view(points, x_range=(0, 60), y_range=(-30, 30), res=0.25, z_range=(G - 0.5, G + 6)):
    """Return (count grid, max-height grid) of shape (ny, nx); row 0 = largest y (left)."""
    m = ((points[:, 2] >= z_range[0]) & (points[:, 2] <= z_range[1]) &
         (points[:, 0] >= x_range[0]) & (points[:, 0] < x_range[1]) &
         (points[:, 1] >= y_range[0]) & (points[:, 1] < y_range[1]))
    p = points[m]
    nx = int((x_range[1] - x_range[0]) / res); ny = int((y_range[1] - y_range[0]) / res)
    ix = ((p[:, 0] - x_range[0]) / res).astype(int)
    iy = ny - 1 - ((p[:, 1] - y_range[0]) / res).astype(int)
    count = np.zeros((ny, nx)); np.add.at(count, (iy, ix), 1)
    height = np.full((ny, nx), np.nan); hz = p[:, 2] - G
    order = np.argsort(hz)                       # later writes win -> max height
    height[iy[order], ix[order]] = hz[order]
    return count, height

count, height = birds_eye_view(points)
ext = [0, 60, -30, 30]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im0 = axes[0].imshow(np.log1p(count), cmap='magma', extent=ext, origin='upper')
axes[0].set_title('BEV: log(1 + points per 0.25 m cell)'); fig.colorbar(im0, ax=axes[0], label='log(1 + count)')
im1 = axes[1].imshow(height, cmap='viridis', extent=ext, origin='upper', vmin=0, vmax=4)
axes[1].set_title('BEV: maximum height above ground'); fig.colorbar(im1, ax=axes[1], label='Height (m)')
for ax in axes:
    ax.set_xlabel('x, forward (m)'); ax.set_ylabel('y, left (m)'); ax.grid(False)
plt.tight_layout(); plt.show()

occ = np.nan_to_num(height) > 0.3
print("--- BEV (text summary) ---")
print(f"  Grid {count.shape[1]} x {count.shape[0]} cells of 0.25 m; occupied cells (>0.3 m above ground): {occ.sum():,}")
print("  Objects appear as bright blobs with empty 'shadows' behind them (occlusion);")
print("  BEV is the common representation for detection, tracking, free-space estimation and planning.")

---
## Part 4: Camera vs LiDAR on the same scene

We render a **camera image** of the same synthetic scene with a pinhole model (one ray per pixel, coloured by object class and shaded by distance) and compare it with the LiDAR returns projected into the same image. The camera is dense but has no direct depth; the LiDAR has precise depth but is sparse.

In [ ]:
W, H, f = 480, 200, 260.0                      # image size [px] and focal length [px]
cx, cy = W / 2, H / 2 + 20
u, v = np.meshgrid(np.arange(W) + 0.5, np.arange(H) + 0.5)
d = np.stack([np.ones_like(u), -(u - cx) / f, -(v - cy) / f], -1).reshape(-1, 3)
d /= np.linalg.norm(d, axis=1, keepdims=True)
t_img, id_img = ray_boxes(np.zeros(3), d, max_range=150)

class_rgb = {'car': (0.0, 0.45, 0.70), 'truck': (0.84, 0.37, 0.0), 'pedestrian': (0.80, 0.47, 0.65),
             'building': (0.6, 0.6, 0.6), 'ground': (0.35, 0.35, 0.35), 'sky': (0.75, 0.88, 0.97)}
img = np.zeros((len(d), 3))
for k, nme in enumerate(names):
    img[id_img == k] = class_rgb[CLASS_OF[nme]]
img[id_img == -1] = class_rgb['ground']; img[id_img == -2] = class_rgb['sky']
shade = np.where(np.isfinite(t_img), 1.0 / (1.0 + t_img / 60.0), 1.0)
img = (img * shade[:, None]).reshape(H, W, 3)

# Project LiDAR points (x forward) into the same image
front = points[:, 0] > 0.5
pf = points[front]
pu = cx - f * pf[:, 1] / pf[:, 0]; pv = cy - f * pf[:, 2] / pf[:, 0]
inside = (pu >= 0) & (pu < W) & (pv >= 0) & (pv < H)

fig, axes = plt.subplots(2, 1, figsize=(12, 9))
axes[0].imshow(img); axes[0].set_title('Synthetic camera view (dense; colour = class, brightness = distance)')
axes[1].imshow(np.ones_like(img) * 0.15)
sc = axes[1].scatter(pu[inside], pv[inside], c=np.linalg.norm(pf[inside], axis=1), cmap='viridis', s=1)
axes[1].set_xlim(0, W); axes[1].set_ylim(H, 0)
axes[1].set_title('LiDAR points projected into the same image (sparse; colour = range)')
fig.colorbar(sc, ax=axes[1], label='Range (m)', fraction=0.025)
for ax in axes:
    ax.set_xlabel('u (px)'); ax.set_ylabel('v (px)'); ax.grid(False)
plt.tight_layout(); plt.show()

ped_k = names.index('pedestrian')
ped_px = (id_img == ped_k).sum()
ped_pts = (labels == ped_k).sum()
print("--- Camera vs LiDAR (text summary) ---")
print(f"  Camera: {W*H:,} pixels, every pixel has a colour but no measured depth.")
print(f"  LiDAR: {inside.sum():,} points fall inside the image ({100*inside.sum()/(W*H):.1f} % of pixels), each with a range.")
print(f"  The pedestrian at ~18 m covers {ped_px} camera pixels but only {ped_pts} LiDAR returns.")

---
## Part 5: Radar measures velocity directly

A radar measures the **Doppler shift** of the reflected wave. For a target with radial velocity $v_r$ and carrier wavelength $\lambda = c / f_c$:

$$ f_D = \frac{2 v_r}{\lambda} $$

Automotive radars operate in the 76–81 GHz band (77 GHz → λ ≈ 3.9 mm).

In [ ]:
c = 299_792_458.0
f_c = 77e9
lam = c / f_c
v = np.array([1.0, 5.0, 13.9, 27.8, 36.1])         # m/s (13.9 m/s = 50 km/h, 27.8 = 100 km/h, 36.1 = 130 km/h)
f_d = 2 * v / lam
print(f"Wavelength at 77 GHz: {lam*1e3:.2f} mm")
for vi, fi in zip(v, f_d):
    print(f"  radial velocity {vi:5.1f} m/s ({vi*3.6:5.1f} km/h) -> Doppler shift {fi/1e3:6.2f} kHz")
print("A camera or ToF LiDAR must estimate velocity by differencing positions over frames (noisier, delayed);"
      " radar (and FMCW LiDAR) measure the radial component in a single measurement."
      " The tangential component (crossing traffic) is NOT measured by Doppler.")

### 4D imaging radar

Conventional automotive radars resolve range, azimuth and radial velocity. **4D imaging radars** add **elevation** using larger virtual antenna arrays (MIMO), producing sparse point clouds similar to a low-resolution LiDAR but robust to weather. This helps to separate, for example, an overhead sign gantry from a stopped vehicle. Public datasets include K-Radar (Paek et al., NeurIPS 2022 Datasets & Benchmarks), View-of-Delft (Palffy et al., IEEE RA-L 2022) and MAN TruckScenes (Fent et al., NeurIPS 2024 Datasets & Benchmarks; 360° 4D radar coverage).

---
## Part 6: Weather and redundancy

The chart below is a **qualitative** summary of how each modality tends to degrade. It is *not* a measurement; real degradation depends on the product, the intensity of the condition and the algorithm.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
conditions = ['Clear', 'Rain', 'Fog', 'Night', 'Snow']
robustness = {'Camera': [95, 40, 30, 20, 35], 'LiDAR': [95, 75, 50, 95, 60], 'Radar': [90, 95, 85, 90, 90]}
x = np.arange(len(conditions)); w = 0.25
fig, ax = plt.subplots(figsize=(11, 5))
for i, (s, vals) in enumerate(robustness.items()):
    ax.bar(x + (i - 1) * w, vals, w, label=s, color=SENSOR_COLOURS[s])
ax.set_xticks(x); ax.set_xticklabels(conditions)
ax.set_ylabel('Relative robustness (illustrative, 0-100)'); ax.set_xlabel('Condition')
ax.set_ylim(0, 100); ax.legend()
ax.set_title('Illustrative (qualitative) sensor robustness - not measured data')
plt.tight_layout(); plt.show()

print("NOTE: values are illustrative teaching numbers, not measurements.")
for j, cnd in enumerate(conditions):
    top = max(robustness[s][j] for s in robustness)
    best = [s for s in robustness if robustness[s][j] == top]
    print(f"  {cnd}: " + ", ".join(f"{s} {robustness[s][j]}" for s in robustness) + f" -> most robust here: {' / '.join(best)}")

### How much does redundancy buy?

If each of $n$ sensors independently misses an object with probability $p_i$, all of them miss with probability $\prod_i p_i$. Real sensors share **common causes** (snow covering the whole front, a shared power supply, the same software stack). A simple way to model this is the **β-factor model** from reliability engineering: a fraction β of each sensor's miss probability is common to all sensors.

> **Illustrative data:** the miss probabilities and β values below are synthetic, chosen for teaching.

In [ ]:
p_miss = {'Camera': 0.05, 'LiDAR': 0.03, 'Radar': 0.08}    # illustrative per-object miss probabilities

def p_all_miss(ps, beta):
    """beta-factor model (rare-event approximation): a common-cause part Q_cc = beta * min(p)
    defeats all sensors at once; the remaining parts (p_i - Q_cc) fail independently."""
    ps = np.asarray(ps, float)
    q_cc = beta * ps.min()
    return q_cc + np.prod(ps - q_cc)

combos = [['Camera'], ['Camera', 'LiDAR'], ['Camera', 'LiDAR', 'Radar']]
betas = [0.0, 0.01, 0.1]
fig, ax = plt.subplots(figsize=(9, 5))
for b, col in zip(betas, OKABE_ITO[4:7]):
    vals = [p_all_miss([p_miss[s] for s in cmb], b) for cmb in combos]
    ax.semilogy(range(1, 4), vals, 'o-', color=col, label=f'beta = {b}')
    for k, val in enumerate(vals):
        print(f"  beta={b:<5} sensors={'+'.join(combos[k]):22s} P(all miss) = {val:.2e}")
ax.set_xticks([1, 2, 3]); ax.set_xticklabels(['Camera', '+LiDAR', '+Radar'])
ax.set_xlabel('Sensors combined'); ax.set_ylabel('P(all sensors miss) (log scale)')
ax.set_title('Redundancy with common-cause failures (illustrative)'); ax.legend()
plt.tight_layout(); plt.show()

gain_indep = p_all_miss([p_miss[s] for s in combos[0]], 0) / p_all_miss([p_miss[s] for s in combos[2]], 0)
gain_cc = p_all_miss([p_miss[s] for s in combos[0]], 0.1) / p_all_miss([p_miss[s] for s in combos[2]], 0.1)
print(f"Adding LiDAR and radar reduces P(all miss) by {gain_indep:,.0f}x if failures are independent, "
      f"but only by {gain_cc:.0f}x with beta = 0.1.")
print("Common-cause failures (snow/ice on all sensors, shared power or software) dominate the residual risk.")

---
## Safety perspective

- Each modality has **performance limitations** that are not faults: a camera blinded by low sun or a LiDAR attenuated by fog works exactly as designed. Under ISO 21448:2022 these are *performance insufficiencies*, and the conditions that reveal them (glare, fog, spray, retro-reflective signs) are *triggering conditions*.
- Diverse sensors **support** a safety argument — for ISO 26262 (random hardware faults, ASIL decomposition into sufficiently independent elements per ISO 26262-9) and for ISO 21448 (mitigating performance insufficiencies) — but only if the independence assumption is justified. The β-factor example shows how quickly common causes erode the benefit.
- Neither standard *mandates* a particular sensor set; the required evidence follows from the hazard analysis.

Continue with: [04_SOTIF_ISO21448/02 — Triggering conditions](../04_SOTIF_ISO21448/02_triggering_conditions_and_scenarios.ipynb) and [03_Functional_Safety_ISO26262/01 — ISO 26262 fundamentals](../03_Functional_Safety_ISO26262/01_iso26262_fundamentals.ipynb).

## Key takeaways

- Camera = dense semantics without direct depth; LiDAR = precise but sparse 3D; radar = weather-robust range and radial velocity with low angular resolution.
- 4D imaging radar adds elevation; FMCW LiDAR adds per-point radial velocity — both narrow the traditional gaps between modalities.
- Automotive LiDAR prices have fallen to about US$200 per unit for 2025 mass-production designs (as of 2026); quoted ranges must state the target reflectivity.
- A ray-cast scan shows the real structure of LiDAR data: rings, range-dependent density and occlusion shadows.
- Redundancy helps only as far as failures are independent; common-cause degradation (weather, contamination, shared power/software) dominates residual risk.
- Sensor limitations are SOTIF performance insufficiencies, not faults; diversity supports but does not replace the safety argument.

## Self-assessment

1. Which sensor works best in fog, and why?
2. Why can a conventional radar struggle to distinguish a stopped vehicle from an overhead sign gantry, and what changes with 4D imaging radar?
3. What is a bird's-eye view, and why is it a popular representation for detection and planning?
4. In the simulated scan, why do near objects receive many more LiDAR returns than distant ones?
5. Three sensors each miss an object with probability 0.05. What is P(all miss) if failures are independent? Why is the real value likely higher?
6. Which sensor gives the best object classification, and what is its main weakness?

<details>
<summary>Show answers</summary>

1. Radar: millimetre waves are much less attenuated and scattered by fog droplets than visible or near-infrared light.
2. Conventional radar has little or no elevation resolution, so both appear as strong static returns at similar range and azimuth. 4D imaging radar measures elevation and can separate them by height.
3. A top-down grid on the ground plane with metric axes. Objects do not overlap in BEV as they do in perspective images, and planning naturally operates in this frame.
4. A fixed angular resolution spreads beams apart with distance: the spacing between neighbouring points grows linearly with range, so the number of points on an object falls roughly with 1/d².
5. 0.05³ = 1.25 × 10⁻⁴. Common causes (weather affecting several sensors, contamination, shared power or software) make failures positively correlated, so the joint miss probability is higher.
6. The camera: it captures colour, texture and text. Its main weakness is sensitivity to lighting and weather (night, glare, fog, rain on the lens).

</details>

## References

1. ISO 21448:2022. *Road vehicles — Safety of the intended functionality.* https://www.iso.org/standard/77490.html
2. ISO 26262-9:2018. *Road vehicles — Functional safety — Part 9: ASIL-oriented and safety-oriented analyses.* https://www.iso.org/standard/68391.html
3. ISO 8855:2011. *Road vehicles — Vehicle dynamics and road-holding ability — Vocabulary* (vehicle axis system).
4. Royo, S., Ballesta-Garcia, M. (2019). An Overview of Lidar Imaging Systems for Autonomous Vehicles. *Applied Sciences* 9(19), 4093. https://doi.org/10.3390/app9194093
5. Paek, D.-H., Kong, S.-H., Wijaya, K. T. (2022). K-Radar: 4D Radar Object Detection for Autonomous Driving in Various Weather Conditions. NeurIPS Datasets & Benchmarks. arXiv:2206.08171
6. Palffy, A. et al. (2022). Multi-class Road User Detection with 3+1D Radar in the View-of-Delft Dataset. *IEEE Robotics and Automation Letters*.
7. Fent, F. et al. (2024). MAN TruckScenes: A multimodal dataset for autonomous trucking in diverse conditions. NeurIPS Datasets & Benchmarks. arXiv:2407.07462
8. Reuters (Nov 2024), reported by LEDinside: Hesai to halve LiDAR prices with ATX. https://www.ledinside.com/node/34820

---
**Navigation:** ← [01 SAE levels and ODD](01_sae_levels_and_odd.ipynb) · [Module index](README.md) · [03 LiDAR and point clouds](03_lidar_point_clouds.ipynb) →